# PagedCore: 15-token prefill + one decode step

Enable **Internet** and **T4 GPU**, then **Run All** in a fresh Kaggle session.
The four code cells contain everything: exact source, frozen Python/dependencies,
the retained failed logits, reproduction, tracing, shared-input FP64 controls and ZIP export.
No GitHub push, Input dataset or previous notebook is needed. Only GPU 0 is used.

The prompt is the same **15-ID prefix of `maximum_output`** used in the failed
acceptance case, including BOS. Prefill chooses a token; one decode step appends
that token at absolute position 15 and attends to **16 tokens in one page**.
It computes logits for the following token. This is one decode step, not merely
requesting one generated token from the API.

1. **Setup:** verify source hashes, frozen environment, checkpoint and one T4.
2. **Reproduce:** compare both raw logit tensors exactly with the retained failure.
   A ratio above 1 fails the unchanged logit rule. A mismatch in the reproduction
   control stops tracing so a different failure is not mistaken for this one.
3. **Trace and replay:** find the earliest FP32 difference and earliest difference
   surviving FP16. Check actual KV views, positions, old history, rotated K and
   unrotated V, and exact instrumented-versus-baseline logits. Replay only those
   first differing layers on shared Q/K/V and compare independent CPU FP64 math.
   Shared P/V controls isolate the value product. CPU FP64 is a diagnostic reference;
   the decoder paths and native replay run on T4. Native replay must match before
   interpreting its errors. Layers are numbered **0-21**.
4. **Export:** always retain logs, raw tensors, source hashes and partial failures.
   Download the final ZIP, including on failure, and send it back for analysis.

Why this helps: wrong KV values point toward storage or indexing; equal operands
with different outputs point toward arithmetic. FP64 measures each path's error.
Small primitive differences alone do not justify changing the integrated comparator.
This notebook collects evidence; it does not guess a repair, change thresholds,
close P5.3 or replace the full required T4 suite. The source snapshot records the
base commit plus exact working files, rather than claiming a new clean commit.


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import traceback
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

PAYLOAD = "..."
SNAPSHOT_SHA256 = "271f63ac9dca4856cd1ad5ca5ff3110a6277ef183231c12ae6c5c03c420744e2"


output_override = os.environ.get("PAGEDCORE_EVIDENCE_WORKDIR")
working = (
    Path(output_override).expanduser().resolve()
    if output_override
    else Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
working.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix="pagedcore-layer-trace-", dir=working))
source = run_root / "source"
evidence = run_root / "evidence"
evidence.mkdir()
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(source)
identity = json.loads((source / "source-identity.json").read_text())
environment = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
)
commands = []


def run(name, command, *, required=False):
    record = {"name": name, "argv": list(map(str, command)), "cwd": str(source)}
    commands.append(record)
    print(name, flush=True)
    with (evidence / (name + ".log")).open("w") as log:
        process = subprocess.Popen(
            command, cwd=source, env=environment, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    print(name + " is running; progress is saved in " + str(log.name), flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            record["interrupted"] = True
            raise
    record["returncode"] = process.returncode
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    print(name, "exit", process.returncode, flush=True)
    if required and process.returncode:
        raise RuntimeError(name + " failed; inspect its retained log")


def verify_source():
    checks = {
        name: hashlib.sha256((source / name).read_bytes()).hexdigest()
        for name in identity["sha256"]
    }
    assert checks == identity["sha256"], "source changed after notebook generation"
    return checks


setup_ok = False
try:
    verify_source()
    run("setup-uv", [sys.executable, "-m", "pip", "install", "uv==0.11.27"], required=True)
    run(
        "setup-frozen",
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--extra",
            "cuda",
            "--extra",
            "dev",
            "--python",
            (source / ".python-version").read_text().strip(),
        ],
        required=True,
    )
    python = source / ".venv/bin/python"
    run(
        "resolved-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
    )
    run("gpu-inventory", ["nvidia-smi", "-q"])

    target_code = "import json,sys,torch; from pathlib import Path; from pagedcore.cli import collect_environment_diagnostics; assert torch.cuda.is_available() and torch.cuda.device_count()==1; assert 'T4' in torch.cuda.get_device_name(0); assert torch.__version__=='2.7.1+cu126'; p=torch.cuda.get_device_properties(0); Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,name=p.name,uuid=str(p.uuid),capability=[p.major,p.minor],visible_devices=torch.cuda.device_count(),total_memory=p.total_memory,environment=collect_environment_diagnostics()),indent=2))"
    run("target", [str(python), "-c", target_code, str(evidence / "target.json")], required=True)
    run(
        "checkpoint-verify",
        [
            str(python),
            "-m",
            "scripts.verify_owned_checkpoint",
            "--output",
            str(evidence / "environment-manifest.json"),
        ],
        required=True,
    )
    setup_ok = True
except BaseException:
    (evidence / "setup-failure.txt").write_text(traceback.format_exc())
    print("Setup incomplete. Run the final cell to download its logs.")

In [ ]:
reproduction_ok = False
try:
    assert setup_ok, "Setup did not complete"
    run(
        "reproduction",
        [
            str(python),
            "-m",
            "scripts.probe_two_token_decode",
            "--prompt-tokens",
            "15",
            "--single-only",
            "--output-dir",
            str(evidence / "reproduction"),
        ],
        required=True,
    )
    exact_check = """
import json, sys, torch
from pathlib import Path
root, output = map(Path, sys.argv[1:])
prior = torch.load(root / 'prior-failure/target.pt', map_location='cpu', weights_only=True)
current = torch.load(output / 'reproduction/single/target.pt', map_location='cpu', weights_only=True)
controls = {name + '_exact': torch.equal(prior[name], current[name]) for name in prior}
(output / 'prior-failure-controls.json').write_text(json.dumps(controls, indent=2))
assert all(controls.values()), 'Raw logits differ from retained T4 failure; inspect environment/source'
"""
    run(
        "prior-failure-control",
        [str(python), "-c", exact_check, str(source), str(evidence)],
        required=True,
    )
    result = json.loads((evidence / "reproduction/single/report.json").read_text())
    print("Attended tokens:", result["attended_tokens"])
    print(
        "Logit tolerance ratio:",
        result["owned_reference"]["max_tolerance_ratio"],
        "(above 1 fails)",
    )
    print("Greedy choice agrees:", result["owned_reference"]["greedy_matches"])
    assert result["discrepancy_reproduced"], "The known failure did not reproduce"
    reproduction_ok = True
except BaseException:
    (evidence / "reproduction-failure.txt").write_text(traceback.format_exc())
    print("Reproduction incomplete. Run the final cell to download its logs.")

In [ ]:
try:
    assert reproduction_ok, "The retained failure must reproduce before interpretation"
    run(
        "layer-trace",
        [
            str(python),
            "-m",
            "scripts.trace_two_token_decode",
            "--prompt-tokens",
            "15",
            "--baseline-dir",
            str(evidence / "reproduction/single"),
            "--output-dir",
            str(evidence / "layer-trace"),
        ],
        required=True,
    )
    summary = json.loads((evidence / "layer-trace/summary.json").read_text())
    result = summary["trace"]
    for name in ("first_fp32_difference", "first_fp16_difference"):
        row = result.get(name)
        print(
            name,
            None
            if row is None
            else {
                k: row[k]
                for k in ("layer", "operation", "max_absolute_error", "first_unequal_index")
            },
        )
    print("Instrumentation:", result["instrumentation_controls"])
    print("Old history unchanged:", result["old_history_unchanged"])
    print("Positions correct:", result["absolute_positions_correct"])
    print("K/V measurements:", result["kv_measurement"])
    print("Native replay exact:", summary["common_input_replay"]["all_native_replays_exact"])
except BaseException:
    (evidence / "trace-failure.txt").write_text(traceback.format_exc())
    print("Trace incomplete. Run the final cell to download its partial evidence.")

In [ ]:
try:
    (evidence / "source-after.json").write_text(json.dumps(verify_source(), indent=2))
except BaseException:
    (evidence / "source-check-failure.txt").write_text(traceback.format_exc())
(evidence / "commands.json").write_text(json.dumps(commands, indent=2))
(evidence / "notebook-run.json").write_text(
    json.dumps(
        {
            "snapshot_sha256": SNAPSHOT_SHA256,
            "base_revision": identity["base_revision"],
            "scope": "15-token prefill + one decode step; cause localization, not acceptance",
            "environment_overrides": {
                k: environment[k]
                for k in (
                    "CUDA_VISIBLE_DEVICES",
                    "CUBLAS_WORKSPACE_CONFIG",
                    "OMP_NUM_THREADS",
                    "MKL_NUM_THREADS",
                )
            },
        },
        indent=2,
    )
)
files = {
    p.relative_to(evidence).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in evidence.rglob("*")
    if p.is_file()
}
(evidence / "checksums.json").write_text(json.dumps(files, indent=2))
archive_path = working / (run_root.name + ".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("source-snapshot.zip", payload)
    for p in evidence.rglob("*"):
        if p.is_file():
            archive.write(p, "evidence/" + p.relative_to(evidence).as_posix())
print("Download and send back:", archive_path.name)
print("If the link fails, download the ZIP from Kaggle's Output file panel.")
display(
    FileLink(
        str(archive_path.relative_to(Path.cwd()))
        if archive_path.is_relative_to(Path.cwd())
        else str(archive_path)
    )
)